# Feature Engineering
>- In this notebook we are converting the raw data into features that our model can understand and learn from
>- The final output will be used in modelling

In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from xgboost import XGBClassifier

In [10]:
#Loading Data
df = pd.read_csv('../Data/Raw Data/2019-Nov.csv')
df = df.fillna('unknown')

df['event_time'] = pd.to_datetime(df['event_time'])
df['event_hour'] = df['event_time'].dt.hour
df['event_min'] = df['event_time'].dt.minute
df['category'] = df['category_code'].str.split('.').str[0]


print(df.shape)
df.head(3)

(1048575, 12)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,event_hour,event_min,category
0,2019-11-01 00:00:00+00:00,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,0,0,electronics
1,2019-11-01 00:00:00+00:00,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283,0,0,appliances
2,2019-11-01 00:00:01+00:00,view,17302664,2053013553853497655,unknown,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387,0,0,unknown


# USER PRODUCT PAIRING

In [19]:
#Extracting features
pair_features =  df.groupby(['user_id','product_id']).agg(
    interactions = ('event_type','count'),
    carts = ('event_type',lambda x:(x=='cart').sum()),
    views = ('event_type',lambda x:(x=='view').sum()),
    avg_seconds = ('event_time',lambda x:(x.dt.hour*3600+x.dt.minute*60+x.dt.second).mean()),
    avg_price = ('price','mean'),
    purchased = ('event_type',lambda x:(x=='purchase').any())
).reset_index()
print('Total user-product pairs : ', len(pair_features))
print('Purchase rate : ',round(pair_features['purchased'].mean() * 100, 2),'%')

Total user-product pairs :  637267
Purchase rate :  2.54 %


**SCOPE**
>- Each row is a unique user-product pair
>- Very few users actually purchased a product, most just viewed
>- If a user added a product to cart there is some higher chance they will buy it

# USER LEVEL FEATURES
>- We look at each user's overall behavior across all products
>- A user who purchases frequently overall is more likely to purchase a new product too

In [ ]:
#Extracting user level features
user_features = df.groupby('user_id').agg(
    total_views = ('event_type',lambda x:(x=='view').sum()),
    total_purchases = ('event_type',lambda x:(x=='purchase').sum()),
    total_carts = ('event_type',lambda x:(x=='cart').sum()),
    products = ('product_id','nunique'),
    sessions = ('user_session','nunique') 
).reset_index()

#Chances of this user buying a product compared to browsing
user_features['purchase_rate'] = (user_features['total_purchases']/user_features['total_views'].replace(0,1)*100).round(2)
print(user_features.shape)
user_features.head(3)

(176639, 7)


,user_id,total_views,total_purchases,total_carts,products,sessions,purchase_rate
0,274969076,3,0,0,1,2,0.0
1,275256741,1,0,0,1,1,0.0
2,295643776,8,0,0,4,2,0.0


**SCOPE**
>- User's purchase_rate tells us how serious a buyer the user is
>- Users with high total carts but low purchases are window shoppers
>- Unique products shows how widely a user browses the catalog

# PRODUCT LEVEL FEATURES
>- We look at how popular each product is across all users.
>- A product that many users have purchased is a stronger recommendation

In [20]:
#Extracting product level features
product_features = df.groupby('product_id').agg(
    views = ('event_type',lambda x:(x=='view').sum()),
    carts = ('event_type',lambda x:(x=='cart').sum()),
    purchases = ('event_type',lambda x:(x=='purchase').sum()),
    users = ('user_id','nunique')
).reset_index()

#Conversion rate of product 
product_features['purchase_rate'] = (product_features['purchases']/product_features['views'].replace(0,1)*100).round(2)

print(product_features.shape)
product_features.head()

(67354, 6)


,product_id,views,carts,purchases,users,purchase_rate
0,1000978,12,0,0,7,0.00
1,1001588,48,0,0,27,0.00
2,1002098,159,0,2,106,1.26
3,1002099,102,0,0,63,0.00
4,1002100,159,0,1,116,0.63


**SCOPE**
>- Products with a low conversion rate have very low chances to be purchased
>- Some products have high conversion rate which means they are strong candidates for recommendation
>- Product users shows how popular a product is across different users

# CATEGORY & BRAND ENCODING
>- Category and brand are text columns, so we convert them to numbers using LabelEncoder
>- We also add category and brand level conversion rates as extra features
>- Because dominating brands like apple could be a good candidate for recommendation  

In [30]:
#Extracting Product,category & brand level information
product_info = df[['product_id', 'category', 'brand']].drop_duplicates(subset='product_id')

#Category level conversion rate
cat_stats = df.groupby('category').agg(
    purchases = ('event_type',lambda x:(x=='purchase').sum()),
    views = ('event_type',lambda x:(x=='view').sum())
).reset_index()

cat_stats['cat_purchase_rate'] = (cat_stats['purchases']/cat_stats['views'].replace(0,1)*100).round(2) 

#Brand level conversion rate
brand_stats = df.groupby('brand').agg(
    purchases = ('event_type',lambda x:(x=='purchase').sum()),
    views = ('event_type',lambda x:(x=='view').sum())
).reset_index()

brand_stats['brand_purchase_rate'] = (brand_stats['purchases']/brand_stats['views'].replace(0,1)*100).round(2) 

product_info = product_info.merge(cat_stats[['category','cat_purchase_rate']],on='category',how='left')
product_info = product_info.merge(brand_stats[['brand','brand_purchase_rate']],on='brand',how='left')

#Encoding
from sklearn.preprocessing import LabelEncoder
cat_encoder = LabelEncoder()
brand_encoder = LabelEncoder()

product_info['category_encoded'] = cat_encoder.fit_transform(product_info['category'])
product_info['brand_encoded'] = brand_encoder.fit_transform(product_info['brand'])

print(product_info.shape)
product_info.head(3)

(67354, 7)


,product_id,category,brand,cat_purchase_rate,brand_purchase_rate,category_encoded,brand_encoded
0,1003461,electronics,xiaomi,2.79,1.82,7,2609
1,5000088,appliances,janome,1.49,0.85,2,1185
2,17302664,unknown,creed,1.42,1.42,13,555


**SCOPE**
>- LabelEncoder converts each unique category/brand to a unique integer
>- Purchase_rates add extra context about how well each category and brand converts

# Merging All Features
>- We combine all the feature tables into one single dataframe — one row per user-product pair with all features attached.

In [61]:
data = pair_features.merge(user_features,on='user_id',how='left')
data = data.merge(product_features,on='product_id',how='left')
data = data.merge(product_info[['product_id','cat_purchase_rate','brand_purchase_rate',
                  'category_encoded','brand_encoded']])

data.dtypes
data['purchased'] = data['purchased'].astype(int)

#Final dataset report
print(f'Final dataset shape : {data.shape}')
print(f'Total Null Values : {sum(data.isna().sum())}')
print(f'Feature Names : {data.columns.tolist()}')
data.head(3)

Final dataset shape : (637267, 23)
Total Null Values : 0
Feature Names : ['user_id', 'product_id', 'interactions', 'carts_x', 'views_x', 'avg_seconds', 'avg_price', 'purchased', 'total_views', 'total_purchases', 'total_carts', 'products', 'sessions', 'purchase_rate_x', 'views_y', 'carts_y', 'purchases', 'users', 'purchase_rate_y', 'cat_purchase_rate', 'brand_purchase_rate', 'category_encoded', 'brand_encoded']


,user_id,product_id,interactions,carts_x,views_x,avg_seconds,avg_price,purchased,total_views,total_purchases,...,purchase_rate_x,views_y,carts_y,purchases,users,purchase_rate_y,cat_purchase_rate,brand_purchase_rate,category_encoded,brand_encoded
0,274969076,1003746,3,0,3,24042.333333,304.75,0,3,0,...,0.0,8,0,0,3,0.00,2.79,1.54,7,2227
1,275256741,1306265,1,0,1,8583.000000,1415.48,0,1,0,...,0.0,8,0,0,5,0.00,1.13,1.03,4,1093
2,295643776,5100337,1,0,1,11753.000000,317.67,0,8,0,...,0.0,793,23,24,511,3.03,2.79,3.70,7,130


**SCOPE**
>- We have merged all the product, user & pair features in a single dataframe
>- We have Replaced the boolean values with integers
>- This dataframe is now ready to be fed directly to the model

# SAVING THE PROCESSED DATA

In [64]:
data.to_csv('../Data/Processed Data/processed_data.csv',index=False)